In [ ]:
import os
import joblib
import glob
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.under_sampling import RandomUnderSampler

# ==========================================
# 1. DATA INGESTION & CLEANING
# ==========================================
DATASET_PATH = 'datasets/CIC-IDS-2017/*.csv'

print(f"Loading data from: {DATASET_PATH}")
csv_files = glob.glob(DATASET_PATH)

if not csv_files:
    raise ValueError("No CSV files found. Check your path.")

dataframes = []
cols_to_drop = ['Flow ID', 'Source IP', 'Destination IP', 'Timestamp']

for file in csv_files:
    df = pd.read_csv(file, encoding='latin-1', low_memory=False)
    df.columns = df.columns.str.strip()
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
    dataframes.append(df)

master_df = pd.concat(dataframes, ignore_index=True)

# 1. Sanitize
master_df = master_df.replace([np.inf, -np.inf], np.nan)
master_df = master_df.dropna()
master_df['Label'] = master_df['Label'].str.upper().str.strip()

print(f"Total records after cleaning: {len(master_df):,}")

X = master_df.drop(columns=['Label'])
y_multi = master_df['Label']

# 3. Encode Multi-class Labels (For Stage 2)
le = LabelEncoder()
y_multi_encoded = le.fit_transform(y_multi)

# 4. Create Binary Labels (For Stage 1)
# 0 = BENIGN, 1 = Any Attack
y_binary = (y_multi != 'BENIGN').astype(int)

# 5. Split Data (Stratify on the Multi-class label to keep rare attacks in test set)
# We split X once, and split the two target arrays using the same indices
X_train, X_test, y_multi_train, y_multi_test, y_bin_train, y_bin_test = train_test_split(
    X, y_multi_encoded, y_binary,
    test_size=0.2,
    random_state=42,
    stratify=y_multi_encoded
)

# 6. Encode Categorical Features (Protocol, etc.)
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

print(f"\nTraining set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

joblib.dump(list(X_train.columns), '../Models/ids17/train_columns.joblib')

print("✅ IDS2017 Models and artifacts saved successfully!")

In [ ]:
# ==========================================
# STAGE 1: BINARY TRIAGE (Benign vs Attack)
# ==========================================
print("\n" + "="*50)
print("STAGE 1: TRAINING BINARY TRIAGE MODEL")
print("="*50)

stage1_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=25,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
stage1_model.fit(X_train, y_bin_train)

stage1_preds = stage1_model.predict(X_test)

print(classification_report(y_bin_test, stage1_preds, target_names=['BENIGN', 'ATTACK'], zero_division=0))

cm_bin = confusion_matrix(y_bin_test, stage1_preds)
tn, fp, fn, tp = cm_bin.ravel()
print(f"False Positives (False Alarms): {fp}")
print(f"False Negatives (Missed Attacks): {fn}")


In [ ]:
# ==========================================
# STAGE 2: MULTI-CLASS DETECTIVE (Attack Type)
# ==========================================
print("\n" + "="*50)
print("STAGE 2: TRAINING MULTI-CLASS DETECTIVE MODEL")
print("="*50)

# We use RandomUnderSampler to cap the majority classes at 10,000
# This prevents the Detective from being overwhelmed by BENIGN and DDoS
print("Applying Undersampling to balance training data...")
sampling_strategy = {}
class_counts = np.bincount(y_multi_train)
for i in range(len(le.classes_)):
    if class_counts[i] > 10000:
        sampling_strategy[i] = 10000

rus = RandomUnderSampler(
    sampling_strategy=sampling_strategy,
    random_state=42
)

X_train_res, y_multi_train_res = rus.fit_resample(X_train, y_multi_train)

stage2_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
stage2_model.fit(X_train_res, y_multi_train_res)

stage2_preds = stage2_model.predict(X_test)

print(classification_report(y_multi_test, stage2_preds, target_names=le.classes_, zero_division=0))


In [ ]:
import numpy as np

print("Evaluating the Full Chained Pipeline (Vectorized)...")

# 1. Predict ALL test rows using Stage 2 at once.
# This takes about 5-10 seconds, not 47 minutes.
stage2_preds_all_encoded = stage2_model.predict(X_test)
stage2_preds_all_labels = le.inverse_transform(stage2_preds_all_encoded)

# 2. Now, apply the logic: If Stage 1 said Benign, the final prediction is Benign.
# Otherwise, the final prediction is whatever Stage 2 said.
# np.where acts like a vectorized if/else statement.
final_predictions = np.where(stage1_preds == 0, "BENIGN", stage2_preds_all_labels)

# 3. Print the report
print("\n--- Final Chained Report ---")
print(classification_report(
    le.inverse_transform(y_multi_test),
    final_predictions,
    zero_division=0
))